# 03. CDK2 도킹 검증 (redocking) — PDB 3RPR / 리간드 25Z목적1. **redocking 검증**: 25Z를 결정 구조 자리에 다시 도킹해 원래 위치를 재현하는지 확인 (RMSD ≤ 2.0 Å이면 합격)2. **박스 크기 점검**: 속도 측정에서 Vina 점수가 양수(+1.10)로 나온 리간드가 16 Å 박스를 벗어났는지 확인검증 방식은 두 가지로 나눠서 봅니다.- **A. 표준 redocking** — 결정 좌표를 입력으로 넣음 (smina/GNINA는 박스 안에서 시작 위치를 무작위화하므로 관행상 허용)- **B. 엄격한 redocking** — SMILES에서 3D 구조를 새로 만들어 입력 (결정 좌표 정보를 완전히 지움)B가 통과하면 보고서에 더 강한 근거로 쓸 수 있습니다.실행 환경: Colab 무료 (T4). 셀마다 결과를 Drive에 즉시 저장합니다.

## 1. Drive 연결 및 경로 확인

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

In [ ]:
import os, subprocess, shutil, json, textwrapfrom pathlib import PathROOT   = Path('/content/drive/MyDrive/cdk2')TOOLS  = ROOT / 'tools'EXP    = ROOT / 'experiment'SPEED  = EXP / '02_gnina_speed'OUT    = EXP / '03_redocking'OUT.mkdir(parents=True, exist_ok=True)GNINA    = TOOLS / 'gnina'RECEPTOR = SPEED / 'receptor.pdb'XTAL     = SPEED / 'ligand_25Z_crystal.pdb'# 3RPR 도킹 박스 (진행 상황 문서에서 확정된 값)BOX = dict(center_x=92.37, center_y=80.29, center_z=-47.95,           size_x=16.1,    size_y=15.8,    size_z=16.5)for p in [GNINA, RECEPTOR, XTAL]:    print(('OK   ' if p.exists() else 'MISSING ') + str(p))print('\n출력 폴더:', OUT)

In [ ]:
# gnina 실행 권한 확인 후 버전 출력os.chmod(GNINA, 0o755)print(subprocess.run([str(GNINA), '--version'], capture_output=True, text=True).stdout)

In [ ]:
# RDKit (RMSD 계산용)!pip -q install rdkit 2>/dev/null | tail -1from rdkit import Chemfrom rdkit.Chem import AllChem, rdMolAlign, rdMolDescriptorsfrom rdkit import RDLoggerRDLogger.DisableLog('rdApp.*')print('rdkit', Chem.rdBase.rdkitVersion)

## 2. 기준 리간드 준비결정 구조에서 떼어낸 25Z는 PDB 파일이라 결합 차수(bond order) 정보가 없습니다.RMSD 계산은 대칭을 고려한 원자 매칭을 하므로 두 분자의 topology가 맞아야 합니다.순서대로 시도합니다.1. RCSB에서 25Z의 ideal SDF를 받아 template으로 결합 차수 부여2. 실패 시 — 양쪽 분자의 결합 차수를 모두 지우고 **연결성 + 원소**로만 매칭 (대칭 처리는 유지)

In [ ]:
ref_pdb = Chem.MolFromPDBFile(str(XTAL), removeHs=True, sanitize=False)print('결정 리간드 원자 수 (H 제외):', ref_pdb.GetNumAtoms())LIG_CODE = '25Z'template = Noneideal_sdf = OUT / f'{LIG_CODE}_ideal.sdf'if not ideal_sdf.exists():    url = f'https://files.rcsb.org/ligands/download/{LIG_CODE}_ideal.sdf'    r = subprocess.run(['curl', '-sSL', '-o', str(ideal_sdf), url], capture_output=True, text=True)    if ideal_sdf.exists() and ideal_sdf.stat().st_size < 100:        ideal_sdf.unlink()if ideal_sdf.exists():    template = Chem.MolFromMolFile(str(ideal_sdf))    if template is not None:        print('RCSB ideal SDF 사용 →', Chem.MolToSmiles(Chem.RemoveHs(template)))    else:        print('ideal SDF를 읽지 못했습니다. fallback 경로를 사용합니다.')else:    print('RCSB 다운로드 실패. fallback 경로를 사용합니다.')

In [ ]:
def flatten(mol):    # 결합 차수·방향족성을 지워 연결성만 남긴 사본 (대칭 매칭은 그대로 작동)    m = Chem.RWMol(mol)    for b in m.GetBonds():        b.SetBondType(Chem.BondType.SINGLE)        b.SetIsAromatic(False)    for a in m.GetAtoms():        a.SetIsAromatic(False)        a.SetFormalCharge(0)        a.SetNoImplicit(True)        a.SetNumExplicitHs(0)    m = m.GetMol()    Chem.SanitizeMol(m, Chem.SanitizeFlags.SANITIZE_SYMMRINGS |                        Chem.SanitizeFlags.SANITIZE_ADJUSTHS,                     catchErrors=True)    return mdef build_reference():    # 기준 분자와 RMSD 매칭 모드를 반환    if template is not None:        try:            from rdkit.Chem import AllChem as _AC            ref = _AC.AssignBondOrdersFromTemplate(Chem.RemoveHs(template), ref_pdb)            Chem.SanitizeMol(ref)            return ref, 'template'        except Exception as e:            print('template 부여 실패:', e)    return flatten(ref_pdb), 'flatten'REF, RMS_MODE = build_reference()print('RMSD 매칭 모드:', RMS_MODE)print('기준 분자 원자 수:', REF.GetNumAtoms())ref_out = OUT / 'ref_25Z.sdf'w = Chem.SDWriter(str(ref_out)); w.write(REF); w.close()print('저장:', ref_out)

## 3. RMSD 계산 함수`rdMolAlign.CalcRMS`를 씁니다. `GetBestRMS`와 달리 **분자를 정렬하지 않고** 제자리에서대칭을 고려한 RMSD를 계산합니다. 도킹 검증에서 필요한 것은 이쪽입니다.두 값을 모두 기록합니다.- **top-pose RMSD** — 점수 1위 pose의 RMSD. 실제 스크리닝에서 우리가 쓰게 되는 값.- **best-pose RMSD** — 9개 pose 중 가장 좋은 값. 탐색이 정답을 찾았는지 여부.

In [ ]:
def pose_rmsds(sdf_path, ref=REF, mode=RMS_MODE):    supp = Chem.SDMolSupplier(str(sdf_path), removeHs=True, sanitize=(mode == 'template'))    rows = []    for i, m in enumerate(supp):        if m is None:            rows.append(dict(pose=i+1, rmsd=None, vina=None, cnn_score=None, cnn_affinity=None))            continue        probe = m if mode == 'template' else flatten(m)        try:            rmsd = rdMolAlign.CalcRMS(probe, ref)        except Exception:            rmsd = None        def prop(name):            return float(m.GetProp(name)) if m.HasProp(name) else None        rows.append(dict(pose=i+1, rmsd=rmsd,                         vina=prop('minimizedAffinity'),                         cnn_score=prop('CNNscore'),                         cnn_affinity=prop('CNNaffinity')))    return rowsdef run_gnina(ligand, out_sdf, box=BOX, exhaustiveness=8, num_modes=9, seed=1, log=None):    cmd = [str(GNINA), '-r', str(RECEPTOR), '-l', str(ligand), '-o', str(out_sdf),           '--exhaustiveness', str(exhaustiveness), '--num_modes', str(num_modes),           '--seed', str(seed)]    for k, v in box.items():        cmd += [f'--{k}', str(v)]    res = subprocess.run(cmd, capture_output=True, text=True)    if log:        Path(log).write_text(res.stdout + '\n--- stderr ---\n' + res.stderr)    if res.returncode != 0:        print(res.stdout[-2000:]); print(res.stderr[-2000:])        raise RuntimeError('gnina 실패')    return res

## 4. A. 표준 redocking (결정 좌표 입력)

In [ ]:
import timelig_A  = OUT / 'input_A_crystal.sdf'w = Chem.SDWriter(str(lig_A)); w.write(REF); w.close()out_A = OUT / 'redock_A.sdf't0 = time.time()run_gnina(lig_A, out_A, log=OUT / 'redock_A.log')print(f'소요 {time.time()-t0:.1f}초')rows_A = pose_rmsds(out_A)for r in rows_A:    print(r)

## 5. B. 엄격한 redocking (SMILES에서 3D 새로 생성)결정 좌표를 전혀 쓰지 않습니다. ETKDG로 conformer를 만들고 MMFF로 완화한 뒤 도킹합니다.

In [ ]:
# 결합 차수를 아는 분자에서만 SMILES를 얻을 수 있습니다src = REF if RMS_MODE == 'template' else templatesmiles_25Z = Chem.MolToSmiles(Chem.RemoveHs(src)) if src is not None else Noneprint('25Z SMILES:', smiles_25Z)lig_B = OUT / 'input_B_fresh3d.sdf'ok_B = Falseif smiles_25Z:    m = Chem.AddHs(Chem.MolFromSmiles(smiles_25Z))    params = AllChem.ETKDGv3(); params.randomSeed = 42    if AllChem.EmbedMolecule(m, params) == 0:        AllChem.MMFFOptimizeMolecule(m, maxIters=2000)        m = Chem.RemoveHs(m)        w = Chem.SDWriter(str(lig_B)); w.write(m); w.close()        ok_B = True        print('새 3D conformer 생성 완료')    else:        print('conformer 생성 실패 — B는 건너뜁니다')else:    print('SMILES를 확보하지 못해 B는 건너뜁니다 (A 결과만 사용)')

In [ ]:
rows_B = Noneif ok_B:    out_B = OUT / 'redock_B.sdf'    t0 = time.time()    run_gnina(lig_B, out_B, log=OUT / 'redock_B.log')    print(f'소요 {time.time()-t0:.1f}초')    rows_B = pose_rmsds(out_B)    for r in rows_B:        print(r)

## 6. 판정 및 저장

In [ ]:
import pandas as pddef summarize(rows, label):    if not rows:        return None    df = pd.DataFrame(rows); df['run'] = label    valid = df.dropna(subset=['rmsd'])    top  = valid.iloc[0]['rmsd'] if len(valid) else None    best = valid['rmsd'].min() if len(valid) else None    return df, top, bestres = {}frames = []for rows, label in [(rows_A, 'A_crystal_input'), (rows_B, 'B_fresh_3d')]:    s = summarize(rows, label)    if s:        df, top, best = s        frames.append(df)        res[label] = dict(top_pose_rmsd=top, best_pose_rmsd=best)tbl = pd.concat(frames, ignore_index=True)tbl.to_csv(OUT / '03_redock_poses.csv', index=False)print('=' * 58)for k, v in res.items():    t, b = v['top_pose_rmsd'], v['best_pose_rmsd']    verdict = '합격' if (t is not None and t <= 2.0) else '재검토 필요'    print(f'{k:18s} top={t:.2f} Å  best={b:.2f} Å  → {verdict}' if t is not None          else f'{k:18s} RMSD 계산 실패')print('=' * 58)print('기준: top-pose RMSD ≤ 2.0 Å')print('저장:', OUT / '03_redock_poses.csv')

## 7. 박스 크기 점검 — Vina 점수가 양수였던 리간드속도 측정 20개 중 점수가 양수로 나온 분자를 찾아, 같은 분자를 **큰 박스**로 다시 도킹해 비교합니다.점수가 음수로 돌아오면 박스가 작아서 생긴 문제이고, 그대로 양수면 분자 자체가 이 포켓에 맞지 않는 것입니다.

In [ ]:
timing = pd.read_csv(SPEED / '02_gnina_timing.csv')print('컬럼:', list(timing.columns))timing.head()

In [ ]:
# 컬럼 이름이 다르면 아래 두 변수만 고쳐주세요score_col  = next((c for c in timing.columns                   if any(k in c.lower() for k in ['affinity', 'vina', 'score'])                   and 'cnn' not in c.lower()), None)smiles_col = next((c for c in timing.columns if 'smiles' in c.lower()), None)id_col     = next((c for c in timing.columns                   if any(k in c.lower() for k in ['chembl', 'id', 'name'])), None)print('점수 컬럼:', score_col, '| SMILES 컬럼:', smiles_col, '| ID 컬럼:', id_col)bad = timing[timing[score_col] > 0] if score_col else timing.iloc[0:0]print(f'\n양수 점수 리간드 {len(bad)}건')bad

In [ ]:
BIG = dict(center_x=BOX['center_x'], center_y=BOX['center_y'], center_z=BOX['center_z'],           size_x=24.0, size_y=24.0, size_z=24.0)box_rows = []for _, row in bad.iterrows():    lid = str(row[id_col]) if id_col else f'lig{_}'    smi = row[smiles_col] if smiles_col else None    if not smi:        print(f'{lid}: SMILES 없음 — 01_chembl_cdk2_activity.csv에서 찾아 넣어주세요')        continue    m = Chem.AddHs(Chem.MolFromSmiles(smi))    p = AllChem.ETKDGv3(); p.randomSeed = 7    if AllChem.EmbedMolecule(m, p) != 0:        print(f'{lid}: conformer 생성 실패'); continue    AllChem.MMFFOptimizeMolecule(m, maxIters=2000)    m = Chem.RemoveHs(m)    lp = OUT / f'box_{lid}.sdf'    w = Chem.SDWriter(str(lp)); w.write(m); w.close()    heavy = m.GetNumAtoms()    for tag, box in [('small_16A', BOX), ('large_24A', BIG)]:        op = OUT / f'box_{lid}_{tag}.sdf'        run_gnina(lp, op, box=box, log=OUT / f'box_{lid}_{tag}.log')        poses = pose_rmsds(op)        best = min((q['vina'] for q in poses if q['vina'] is not None), default=None)        box_rows.append(dict(ligand=lid, heavy_atoms=heavy, box=tag,                             best_vina=best, orig_vina=row[score_col]))        print(f'{lid} [{tag}] best Vina = {best}')if box_rows:    bx = pd.DataFrame(box_rows)    bx.to_csv(OUT / '03_box_check.csv', index=False)    display(bx)    print('저장:', OUT / '03_box_check.csv')

## 8. skill/02_docking.md 에 붙일 요약 생성여기서 나온 숫자를 텍스트로 뽑아둡니다. Solar Pro4가 CSV를 직접 읽지 못할 수 있으므로skill 문서에는 **숫자를 텍스트로** 적어두는 것이 원칙입니다.

In [ ]:
lines = []lines.append('### 도킹 검증 (redocking, PDB 3RPR / 리간드 25Z)\n')lines.append(f'- 도킹 박스: center ({BOX["center_x"]}, {BOX["center_y"]}, {BOX["center_z"]}), '             f'size ({BOX["size_x"]}, {BOX["size_y"]}, {BOX["size_z"]})')lines.append('- 설정: exhaustiveness 8, num_modes 9, seed 1, CNN rescore')lines.append(f'- RMSD: rdMolAlign.CalcRMS (정렬 없음, 대칭 고려), 매칭 모드 {RMS_MODE}, 중원자만\n')lines.append('| 검증 | 입력 | top-pose RMSD | 9개 pose 중 최소 | 판정 |')lines.append('|---|---|---|---|---|')labels = {'A_crystal_input': ('A', '결정 좌표'), 'B_fresh_3d': ('B', 'SMILES → 새 3D')}for k, v in res.items():    tag, desc = labels.get(k, (k, ''))    t, b = v['top_pose_rmsd'], v['best_pose_rmsd']    verdict = '합격 (≤ 2.0 Å)' if (t is not None and t <= 2.0) else '재검토'    lines.append(f'| {tag} | {desc} | {t:.2f} Å | {b:.2f} Å | {verdict} |'                 if t is not None else f'| {tag} | {desc} | 실패 | — | — |')if box_rows:    lines.append('\n### 박스 크기 점검\n')    lines.append('| 리간드 | 중원자 | 16 Å 박스 | 24 Å 박스 |')    lines.append('|---|---|---|---|')    for lid, g in pd.DataFrame(box_rows).groupby('ligand'):        s = g.set_index('box')['best_vina'].to_dict()        lines.append(f'| {lid} | {int(g["heavy_atoms"].iloc[0])} | '                     f'{s.get("small_16A")} | {s.get("large_24A")} |')txt = '\n'.join(lines)(OUT / '03_redocking_summary.md').write_text(txt)print(txt)print('\n저장:', OUT / '03_redocking_summary.md')

## 실행 후 할 일1. `experiment/03_redocking/` 폴더를 로컬 `cdk2` 폴더로 내려받기2. `03_redocking_summary.md` 내용을 `skill/02_docking.md`에 속도 측정 결과와 **함께** 작성 (11/7)3. 판정에 따라 분기   - **top-pose RMSD ≤ 2.0 Å** → 박스와 설정 확정, 11/8 필터링 단계로   - **2.0 Å 초과** → 박스 중심 재계산(결정 리간드 중심) 또는 exhaustiveness 16으로 재시도.     exhaustiveness를 올리면 도킹 시간이 늘어나므로 1,000개 일정에 반영해야 함   - **박스 점검에서 24 Å이 음수로 개선** → 박스를 키우되, lead-like 필터(MW ≤ 450,     회전결합 ≤ 7)로 큰 분자가 애초에 걸러지는지 먼저 확인. 필터로 해결되면 16 Å 유지가 유리